In [4]:
# ------------------------------------------------------------------
# EVALUATION NOTEBOOK
# 
# Bu notebook'ta:
# 1) Recommendation modelleri için evaluation setup kuracağız
# 2) Train-test ayrımı yapacağız
# 3) Ranking metriklerini hesaplayacağız
# 4) Farklı modelleri karşılaştıracağız
#
# Bu aşamada özellikle şu modellere zemin hazırlıyoruz:
# - Popularity baseline
# - Content-based filtering
# - Collaborative filtering
# - Hybrid recommendation
# ------------------------------------------------------------------

In [5]:
# ------------------------------------------------------------------
# GEREKLİ KÜTÜPHANELER
# ------------------------------------------------------------------

import pandas as pd
import numpy as np

In [6]:
# ------------------------------------------------------------------
# CF VERİSİNİ YÜKLEME
#
# Evaluation için user-item interaction verisini kullanacağız.
# Burada reviews_cf_last.csv dosyasını yüklüyoruz.
# Bu dosyada her kullanıcı-ürün çifti için tek bir rating var.
# ------------------------------------------------------------------

cf_data = pd.read_csv("../data_interim/reviews_cf_last.csv")

print("cf_data shape:", cf_data.shape)
cf_data.head()

cf_data shape: (1088891, 4)


/var/folders/tm/wc8rn0xj05v33kkcby8sf0fm0000gn/T/ipykernel_29393/622460587.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  cf_data = pd.read_csv("../data_interim/reviews_cf_last.csv")


,author_id,product_id,rating,submission_time
0,538863,P420652,1,2018-11-01
1,549704,P218700,5,2011-04-21
2,557770,P232903,5,2016-02-19
3,561736,P421998,5,2018-07-28
4,561736,P445951,5,2018-07-28


In [7]:
# ------------------------------------------------------------------
# KOLON KONTROLÜ
# ------------------------------------------------------------------

print(cf_data.columns.tolist())

['author_id', 'product_id', 'rating', 'submission_time']


In [8]:
# ------------------------------------------------------------------
# VERİ TİPLERİNİ DÜZELTME
#
# author_id ve product_id kolonlarını string'e çeviriyoruz.
# Böylece karışık tip kaynaklı sorunları engellemiş oluyoruz.
# submission_time kolonunu da datetime formatına çeviriyoruz.
# ------------------------------------------------------------------

cf_data["author_id"] = cf_data["author_id"].astype(str)
cf_data["product_id"] = cf_data["product_id"].astype(str)
cf_data["submission_time"] = pd.to_datetime(cf_data["submission_time"], errors="coerce")

print(cf_data.dtypes)

author_id                  object
product_id                 object
rating                      int64
submission_time    datetime64[ns]
dtype: object


In [9]:
# ------------------------------------------------------------------
# KULLANICI ETKİLEŞİM SAYILARI
#
# Her kullanıcının kaç farklı rating verdiğini hesaplıyoruz.
# Evaluation için çok az etkileşimi olan kullanıcıları kullanmak istemiyoruz.
# ------------------------------------------------------------------

user_interaction_counts = (
    cf_data.groupby("author_id")["product_id"]
    .count()
    .sort_values(ascending=False)
)

print("Toplam kullanıcı sayısı:", user_interaction_counts.shape[0])
print(user_interaction_counts.head(10))

Toplam kullanıcı sayısı: 503216
author_id
1696370280     292
1288462295     204
7375781965     154
1930716686     153
2330399812     151
5060164185     150
1492711942     135
1738022745     135
10690040573    131
12640060683    118
Name: product_id, dtype: int64


In [10]:
# ------------------------------------------------------------------
# EVALUATION İÇİN UYGUN KULLANICILARI SEÇME
#
# En az 5 etkileşimi olan kullanıcıları seçiyoruz.
# Bu eşik sayesinde train-test split daha anlamlı hale geliyor.
# ------------------------------------------------------------------

min_user_interactions = 5

eligible_users = user_interaction_counts[
    user_interaction_counts >= min_user_interactions
].index.tolist()

print("En az 5 etkileşimi olan kullanıcı sayısı:", len(eligible_users))

En az 5 etkileşimi olan kullanıcı sayısı: 40435


In [11]:
# ------------------------------------------------------------------
# SADECE UYGUN KULLANICILARI TUTMA
#
# Evaluation setup'ında sadece en az 5 etkileşimi olan kullanıcıları
# kullanacağız.
# ------------------------------------------------------------------

eval_df = cf_data[cf_data["author_id"].isin(eligible_users)].copy()

print("Evaluation dataframe shape:", eval_df.shape)
eval_df.head()

Evaluation dataframe shape: (370966, 4)


,author_id,product_id,rating,submission_time
5,562130,P380030,5,2016-05-09
6,562130,P381030,5,2014-11-29
7,582399,P420652,5,2018-11-21
8,582399,P421243,4,2019-02-10
12,681955,P482681,5,2022-04-17


In [12]:
# ------------------------------------------------------------------
# RELEVANT ITEM TANIMI
#
# rating >= 4 olan ürünleri "relevant" kabul ediyoruz.
# Yani kullanıcı bu ürünü beğenmiş / uygun bulmuş sayılıyor.
# ------------------------------------------------------------------

eval_df["is_relevant"] = (eval_df["rating"] >= 4).astype(int)

print(eval_df["is_relevant"].value_counts())

is_relevant
1    312975
0     57991
Name: count, dtype: int64


In [13]:
# ------------------------------------------------------------------
# KULLANICI BAŞINA RELEVANT ITEM SAYISI
#
# User-based holdout yapabilmek için kullanıcının en az 1 relevant item'ı
# olması gerekir. Hatta testte 1 ürün saklayacağımız için bu çok önemli.
# ------------------------------------------------------------------

relevant_counts = (
    eval_df.groupby("author_id")["is_relevant"]
    .sum()
    .sort_values(ascending=False)
)

print(relevant_counts.head(10))
print("En az 1 relevant item'ı olan kullanıcı sayısı:", (relevant_counts >= 1).sum())

author_id
1696370280     253
2330399812     151
1288462295     149
7375781965     149
5060164185     145
1492711942     133
10690040573    131
12640060683    118
1930716686     114
1314992825     114
Name: is_relevant, dtype: int64
En az 1 relevant item'ı olan kullanıcı sayısı: 40175


In [14]:
# ------------------------------------------------------------------
# USER-BASED HOLDOUT SPLIT
#
# Amaç:
# Her kullanıcı için 1 adet relevant item'ı test setine almak
# Geri kalan veriyi train seti olarak kullanmak
#
# Böylece:
# Model kullanıcının geçmişine bakarak bu saklanan ürünü tahmin edebiliyor mu?
# bunu test edebileceğiz
# ------------------------------------------------------------------

import numpy as np

train_rows = []
test_rows = []

# Kullanıcı bazında grupla
grouped = eval_df.groupby("author_id")

for user_id, group in grouped:

    # Kullanıcının relevant item'larını al
    relevant_items = group[group["is_relevant"] == 1]

    # Eğer hiç relevant item yoksa skip (zaten çok az olmalı)
    if len(relevant_items) == 0:
        continue

    # Test için 1 tane relevant item seç
    test_sample = relevant_items.sample(n=1, random_state=42)

    # Geri kalanlar train
    train_sample = group.drop(test_sample.index)

    test_rows.append(test_sample)
    train_rows.append(train_sample)

# DataFrame oluştur
train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (329236, 5)
Test shape: (40175, 5)


In [15]:
# ------------------------------------------------------------------
# SPLIT DOĞRULAMA
# ------------------------------------------------------------------

print("Train kullanıcı sayısı:", train_df["author_id"].nunique())
print("Test kullanıcı sayısı:", test_df["author_id"].nunique())

# Her kullanıcı testte sadece 1 item mı?
test_counts = test_df.groupby("author_id").size()

print("Testte max item sayısı:", test_counts.max())
print("Testte min item sayısı:", test_counts.min())

Train kullanıcı sayısı: 40175
Test kullanıcı sayısı: 40175
Testte max item sayısı: 1
Testte min item sayısı: 1


In [16]:
# ------------------------------------------------------------------
# TRAIN SET İLE CF MODELİNİ YENİDEN KURMA
#
# Evaluation adil olsun diye collaborative filtering modelini
# sadece train_df kullanarak kuruyoruz.
# Test setindeki ürünler model eğitiminde kullanılmayacak.
# ------------------------------------------------------------------

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

cf_train_data = train_df.copy()

# --------------------------------------------------------------
# 1) User / item mapping
# --------------------------------------------------------------
user_ids = cf_train_data["author_id"].unique()
item_ids = cf_train_data["product_id"].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}

idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}
idx_to_item = {idx: item_id for item_id, idx in item_to_idx.items()}

# --------------------------------------------------------------
# 2) Index kolonları
# --------------------------------------------------------------
cf_train_data["user_idx"] = cf_train_data["author_id"].map(user_to_idx)
cf_train_data["item_idx"] = cf_train_data["product_id"].map(item_to_idx)

# --------------------------------------------------------------
# 3) Sparse user-item matrix
# --------------------------------------------------------------
user_item_matrix = csr_matrix(
    (
        cf_train_data["rating"],
        (cf_train_data["user_idx"], cf_train_data["item_idx"])
    ),
    shape=(len(user_to_idx), len(item_to_idx))
)

print("Train user-item matrix shape:", user_item_matrix.shape)

# --------------------------------------------------------------
# 4) TruncatedSVD ile latent factor çıkarma
# --------------------------------------------------------------
n_factors = 50

svd = TruncatedSVD(n_components=n_factors, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T

print("User latent matrix shape:", user_factors.shape)
print("Item latent matrix shape:", item_factors.shape)

# --------------------------------------------------------------
# 5) Tahmini skor matrisi
# --------------------------------------------------------------
predicted_scores = user_factors @ item_factors.T

print("Predicted score matrix shape:", predicted_scores.shape)

Train user-item matrix shape: (40175, 2242)
User latent matrix shape: (40175, 50)
Item latent matrix shape: (2242, 50)
Predicted score matrix shape: (40175, 2242)


In [17]:
# ------------------------------------------------------------------
# PRECISION@K (HIT RATE) HESAPLAMA
#
# Her kullanıcı için:
# - model top-K öneri üretir
# - testteki gerçek ürün bu listede mi kontrol edilir
# ------------------------------------------------------------------

def precision_at_k(model_function, train_df, test_df, k=10):

    hits = 0
    total_users = 0

    # test set üzerinden iterate ediyoruz (her user 1 item)
    for idx, row in test_df.iterrows():

        user_id = row["author_id"]
        true_item = row["product_id"]

        # modelden öneri al
        try:
            recs = model_function(user_id, top_n=k)
        except:
            continue

        if recs is None or len(recs) == 0:
            continue

        recommended_items = recs["product_id"].values

        # hit kontrolü
        if true_item in recommended_items:
            hits += 1

        total_users += 1

    precision = hits / total_users

    print(f"Precision@{k}: {precision:.4f}")
    print(f"Hit sayısı: {hits}")
    print(f"Toplam kullanıcı: {total_users}")

    return precision

In [18]:
# ------------------------------------------------------------------
# CF MODEL WRAPPER (DOĞRU EVALUATION SÜRÜMÜ)
#
# Bu sürümde:
# - model train_df ile kurulmuştur
# - öneri listesinden sadece train geçmişindeki ürünler çıkarılır
# - test item çıkarılmaz
# Böylece modelin test item'ı bulma şansı olur
# ------------------------------------------------------------------

def cf_model_wrapper(user_id, top_n=10):

    # Kullanıcı train modelinde yoksa öneri veremeyiz
    if user_id not in user_to_idx:
        return None

    user_idx = user_to_idx[user_id]
    user_scores = predicted_scores[user_idx]

    rec_df = pd.DataFrame({
        "product_id": item_ids,
        "cf_score": user_scores
    })

    # Sadece train set içinde kullanıcının daha önce gördüğü ürünleri çıkar
    train_rated_items = set(
        train_df.loc[train_df["author_id"] == user_id, "product_id"]
    )

    rec_df = rec_df[~rec_df["product_id"].isin(train_rated_items)]
    rec_df = rec_df.sort_values("cf_score", ascending=False)

    return rec_df.head(top_n)

In [19]:
# ------------------------------------------------------------------
# HIZLI TEST İÇİN KÜÇÜK ÖRNEKLEM
#
# Önce evaluation pipeline'ının mantıklı çalışıp çalışmadığını görmek için
# test setin küçük bir kısmında deneyelim.
# ------------------------------------------------------------------

test_df_sample = test_df.head(1000).copy()

precision_at_k(cf_model_wrapper, train_df, test_df_sample, k=10)

Precision@10: 0.3550
Hit sayısı: 355
Toplam kullanıcı: 1000


0.355

In [20]:

# ------------------------------------------------------------------
# POPULARITY BASELINE İÇİN ÜRÜN İSTATİSTİKLERİ
#
# Train set içindeki ürünler için:
# - ortalama rating
# - review sayısı
#
# hesaplıyoruz.
# Bu bilgilerle popularity score oluşturacağız.
# ------------------------------------------------------------------

popularity_stats = (
    train_df.groupby("product_id")
    .agg(
        mean_rating=("rating", "mean"),
        review_count=("rating", "count")
    )
    .reset_index()
)

print("Popularity stats shape:", popularity_stats.shape)
popularity_stats.head()

Popularity stats shape: (2242, 3)


,product_id,mean_rating,review_count
0,P107306,3.625000,40
1,P114902,4.243056,144
2,P12045,4.354839,341
3,P122651,4.173913,23
4,P122661,4.284091,88


In [21]:
# ------------------------------------------------------------------
# POPULARITY SCORE HESAPLAMA
#
# Sadece ortalama rating kullanmak risklidir.
# Çünkü çok az review alan ürünler haksız şekilde öne çıkabilir.
#
# Bu yüzden weighted score kullanıyoruz:
# - global mean
# - review_count
# - m sabiti
# ------------------------------------------------------------------

global_mean = train_df["rating"].mean()
m = 20

popularity_stats["popularity_score"] = (
    (popularity_stats["mean_rating"] * popularity_stats["review_count"] + global_mean * m)
    /
    (popularity_stats["review_count"] + m)
)

popularity_stats = popularity_stats.sort_values(
    "popularity_score",
    ascending=False
)

popularity_stats.head(10)

,product_id,mean_rating,review_count,popularity_score
2120,P503992,4.899522,209,4.851273
2117,P503936,4.846816,1162,4.838360
1764,P481335,4.898551,138,4.828744
1997,P501157,4.884146,164,4.825769
2086,P503668,4.846734,398,4.822827
2127,P504026,4.891304,138,4.822415
1893,P483679,4.840156,513,4.821654
2082,P503651,4.852234,291,4.819748
2129,P504031,4.945205,73,4.816576
2119,P503991,4.884615,130,4.812944


In [22]:
# ------------------------------------------------------------------
# POPULARITY BASELINE RECOMMENDATION FUNCTION
#
# Bu fonksiyon kullanıcıya en popüler ürünleri önerir.
# Kişiselleştirme yapmaz.
#
# Ancak:
# - kullanıcının train geçmişinde zaten gördüğü ürünleri çıkarır
# ------------------------------------------------------------------

def popularity_model_wrapper(user_id, top_n=10):

    # Kullanıcının train set içindeki geçmiş ürünleri
    train_rated_items = set(
        train_df.loc[train_df["author_id"] == user_id, "product_id"]
    )

    rec_df = popularity_stats.copy()

    # Daha önce görülen ürünleri çıkar
    rec_df = rec_df[~rec_df["product_id"].isin(train_rated_items)]

    return rec_df[["product_id", "popularity_score"]].head(top_n)

In [23]:
# ------------------------------------------------------------------
# POPULARITY BASELINE İÇİN HIZLI EVALUATION
# ------------------------------------------------------------------

precision_at_k(popularity_model_wrapper, train_df, test_df_sample, k=10)

Precision@10: 0.0010
Hit sayısı: 1
Toplam kullanıcı: 1000


0.001

In [24]:
# ------------------------------------------------------------------
# CONTENT-BASED EVALUATION İÇİN ÜRÜN TABLOSUNU YÜKLEME
#
# Evaluation notebook ayrı bir runtime olduğu için ürün tablosunu burada
# yeniden yüklüyoruz.
# ------------------------------------------------------------------

products_clean = pd.read_csv("../data_interim/products_clean.csv")

print("products_clean shape:", products_clean.shape)
products_clean.head()

products_clean shape: (8494, 19)


,product_id,product_name,brand_name,loves_count,rating,reviews,ingredients,price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,product_text
0,P473671,Fragrance Discovery Set,19-69,6320,3.6364,11.0,Capri Eau de Parfum: Alcohol Denat. (SD Alcoho...,35.0,0,0,1,0,0,Unisex/ Genderless Scent Warm &Spicy Scent Woo...,Fragrance,Value & Gift Sets,Perfume Gift Sets,0,Fragrance Discovery Set 19-69 Fragrance Value ...
1,P473668,La Habana Eau de Parfum,19-69,3827,4.1538,13.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Warm ...,Fragrance,Women,Perfume,2,La Habana Eau de Parfum 19-69 Fragrance Women ...
2,P473662,Rainbow Bar Eau de Parfum,19-69,3253,4.2500,16.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Woody...,Fragrance,Women,Perfume,2,Rainbow Bar Eau de Parfum 19-69 Fragrance Wome...
3,P473660,Kasbah Eau de Parfum,19-69,3018,4.4762,21.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Warm ...,Fragrance,Women,Perfume,2,Kasbah Eau de Parfum 19-69 Fragrance Women Per...
4,P473658,Purple Haze Eau de Parfum,19-69,2691,3.2308,13.0,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,0,0,1,0,0,Unisex/ Genderless Scent Layerable Scent Woody...,Fragrance,Women,Perfume,2,Purple Haze Eau de Parfum 19-69 Fragrance Wome...


In [25]:
# ------------------------------------------------------------------
# CONTENT-BASED MODEL İÇİN GEREKLİ KOLON KONTROLÜ
# ------------------------------------------------------------------

print("product_text var mı?:", "product_text" in products_clean.columns)
print(products_clean.columns.tolist())

product_text var mı?: True
['product_id', 'product_name', 'brand_name', 'loves_count', 'rating', 'reviews', 'ingredients', 'price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'product_text']


In [26]:
# ------------------------------------------------------------------
# CONTENT-BASED MODELİNİ EVALUATION NOTEBOOK İÇİN YENİDEN KURMA
#
# Bu notebook ayrı çalıştığı için TF-IDF matrix ve similarity matrix
# burada yeniden oluşturuluyor.
# ------------------------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(products_clean["product_text"].fillna(""))

similarity_matrix = cosine_similarity(tfidf_matrix)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

TF-IDF matrix shape: (8494, 5000)
Similarity matrix shape: (8494, 8494)


In [27]:
# ------------------------------------------------------------------
# PRODUCT_ID -> INDEX MAPPING
#
# Similarity matrix satır/sütunları products_clean index'ine göre çalışır.
# Bu yüzden product_id ile index arasında mapping kuruyoruz.
# ------------------------------------------------------------------

productid_to_index = {pid: idx for idx, pid in enumerate(products_clean["product_id"])}
index_to_productid = {idx: pid for pid, idx in productid_to_index.items()}

print("Toplam mapped ürün sayısı:", len(productid_to_index))

Toplam mapped ürün sayısı: 8494


In [28]:
# ------------------------------------------------------------------
# KULLANICI İÇİN SEED ITEM SEÇME
#
# Content-based recommendation için her kullanıcıdan bir referans ürün seçiyoruz.
# Bu ürün train set içindeki relevant item'lar arasından seçilir.
#
# Seçim mantığı:
# - sadece rating >= 4 olan ürünler
# - en yüksek rating
# - eşitlik varsa en güncel submission_time
# ------------------------------------------------------------------

def get_seed_item_for_user(user_id):

    user_train = train_df[train_df["author_id"] == user_id].copy()

    # Sadece relevant train item'lar
    user_relevant = user_train[user_train["rating"] >= 4].copy()

    if user_relevant.empty:
        return None

    # En iyi seed ürünü seç
    user_relevant = user_relevant.sort_values(
        by=["rating", "submission_time"],
        ascending=[False, False]
    )

    seed_item = user_relevant.iloc[0]["product_id"]

    return seed_item

In [29]:
# ------------------------------------------------------------------
# CONTENT-BASED MODEL WRAPPER
#
# Her kullanıcı için:
# 1) Train set içinden bir seed item seç
# 2) O ürüne benzer ürünleri similarity matrix'ten çek
# 3) Kullanıcının train geçmişindeki ürünleri öneri listesinden çıkar
# 4) Top-N öneriyi döndür
# ------------------------------------------------------------------

def content_model_wrapper(user_id, top_n=10):

    # Kullanıcı için seed item seç
    seed_item = get_seed_item_for_user(user_id)

    if seed_item is None:
        return None

    # Seed item products_clean içinde yoksa öneri veremeyiz
    if seed_item not in productid_to_index:
        return None

    seed_index = productid_to_index[seed_item]

    # Seed item için similarity skorlarını al
    similarity_scores = list(enumerate(similarity_matrix[seed_index]))

    # En yüksek benzerlik skoruna göre sırala
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    # İlk eleman ürünün kendisi olduğu için çıkar
    similarity_scores = similarity_scores[1:]

    # Kullanıcının train geçmişindeki ürünleri çıkar
    train_rated_items = set(
        train_df.loc[train_df["author_id"] == user_id, "product_id"]
    )

    recommendation_rows = []

    for idx, score in similarity_scores:
        product_id = index_to_productid[idx]

        if product_id in train_rated_items:
            continue

        recommendation_rows.append({
            "product_id": product_id,
            "similarity_score": score
        })

        if len(recommendation_rows) >= top_n:
            break

    if len(recommendation_rows) == 0:
        return None

    rec_df = pd.DataFrame(recommendation_rows)

    return rec_df

In [30]:
# ------------------------------------------------------------------
# CONTENT WRAPPER TEK KULLANICI TESTİ
# ------------------------------------------------------------------

sample_user_id = test_df_sample.iloc[0]["author_id"]

print("Sample user:", sample_user_id)
print("Seed item:", get_seed_item_for_user(sample_user_id))

content_model_wrapper(sample_user_id, top_n=10)

Sample user: 10000117144
Seed item: P504794


,product_id,similarity_score
0,P504773,0.425748
1,P503642,0.366593
2,P505209,0.352974
3,P438620,0.350142
4,P503652,0.323835
5,P417867,0.323363
6,P504840,0.316693
7,P504784,0.315829
8,P479125,0.306020
9,P446930,0.304633


In [31]:
# ------------------------------------------------------------------
# CONTENT-BASED MODEL İÇİN HIZLI EVALUATION
# ------------------------------------------------------------------

precision_at_k(content_model_wrapper, train_df, test_df_sample, k=10)

Precision@10: 0.1247
Hit sayısı: 122
Toplam kullanıcı: 978


0.12474437627811862

In [32]:
# ------------------------------------------------------------------
# HYBRID MODEL WRAPPER
#
# Her kullanıcı için:
# 1) Train set içinden bir seed item seçilir
# 2) Seed item content-based tarafı besler
# 3) user_id collaborative filtering tarafını besler
# 4) İki skor birleşerek hybrid öneri üretilir
#
# Bu wrapper evaluation fonksiyonuna uygun şekilde yalnızca
# product_id ve final_score döndürür.
# ------------------------------------------------------------------

def hybrid_model_wrapper(user_id, top_n=10, alpha=0.6):

    # Kullanıcı için seed item seç
    seed_item = get_seed_item_for_user(user_id)

    if seed_item is None:
        return None

    # Seed item products_clean içinde yoksa öneri üretemeyiz
    if seed_item not in productid_to_index:
        return None

    # --------------------------------------------------------------
    # 1) Content-based önerileri al
    # --------------------------------------------------------------
    seed_index = productid_to_index[seed_item]

    similarity_scores = list(enumerate(similarity_matrix[seed_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:]  # seed item'ın kendisini çıkar

    train_rated_items = set(
        train_df.loc[train_df["author_id"] == user_id, "product_id"]
    )

    content_rows = []

    for idx, score in similarity_scores:
        product_id = index_to_productid[idx]

        if product_id in train_rated_items:
            continue

        content_rows.append({
            "product_id": product_id,
            "similarity_score": score
        })

        if len(content_rows) >= 50:
            break

    content_rec = pd.DataFrame(content_rows)

    # --------------------------------------------------------------
    # 2) CF önerilerini al
    # --------------------------------------------------------------
    if user_id not in user_to_idx:
        return None

    user_idx = user_to_idx[user_id]
    user_scores = predicted_scores[user_idx]

    cf_rec = pd.DataFrame({
        "product_id": item_ids,
        "cf_score": user_scores
    })

    cf_rec = cf_rec[~cf_rec["product_id"].isin(train_rated_items)]
    cf_rec = cf_rec.sort_values("cf_score", ascending=False).head(50)

    # --------------------------------------------------------------
    # 3) Union-based merge
    # --------------------------------------------------------------
    hybrid_df = content_rec.merge(
        cf_rec,
        on="product_id",
        how="outer"
    )

    if hybrid_df.empty:
        return None

    hybrid_df["similarity_score"] = hybrid_df["similarity_score"].fillna(0)
    hybrid_df["cf_score"] = hybrid_df["cf_score"].fillna(0)

    # --------------------------------------------------------------
    # 4) Min-max normalization
    # --------------------------------------------------------------
    def min_max_normalize(series):
        min_val = series.min()
        max_val = series.max()

        if max_val == min_val:
            return pd.Series([0] * len(series), index=series.index)

        return (series - min_val) / (max_val - min_val)

    hybrid_df["content_score_norm"] = min_max_normalize(hybrid_df["similarity_score"])
    hybrid_df["cf_score_norm"] = min_max_normalize(hybrid_df["cf_score"])

    # --------------------------------------------------------------
    # 5) Final hybrid score
    # --------------------------------------------------------------
    hybrid_df["final_score"] = (
        alpha * hybrid_df["content_score_norm"] +
        (1 - alpha) * hybrid_df["cf_score_norm"]
    )

    hybrid_df = hybrid_df.sort_values("final_score", ascending=False)

    return hybrid_df[["product_id", "final_score"]].head(top_n)

In [33]:
# ------------------------------------------------------------------
# HYBRID WRAPPER TEK KULLANICI TESTİ
# ------------------------------------------------------------------

sample_user_id = test_df_sample.iloc[0]["author_id"]

print("Sample user:", sample_user_id)
print("Seed item:", get_seed_item_for_user(sample_user_id))

hybrid_model_wrapper(sample_user_id, top_n=10, alpha=0.6)

Sample user: 10000117144
Seed item: P504794


,product_id,final_score
92,P504773,0.600000
84,P503642,0.516634
97,P505209,0.497441
30,P438620,0.493449
85,P503652,0.456376
17,P417867,0.455711
95,P504840,0.446310
93,P504784,0.445094
70,P479125,0.431270
36,P446930,0.429315


In [34]:
# ------------------------------------------------------------------
# HYBRID MODEL İÇİN HIZLI EVALUATION
# ------------------------------------------------------------------

precision_at_k(
    lambda user_id, top_n=10: hybrid_model_wrapper(user_id, top_n=top_n, alpha=0.6),
    train_df,
    test_df_sample,
    k=10
)

Precision@10: 0.2485
Hit sayısı: 243
Toplam kullanıcı: 978


0.24846625766871167